# Data validation with Pydantic

A student form checks each answer before it keeps it.

This is lesson 29. The form has a name and an age. A normal class can write `age: int` as a note. Python keeps the note and does not check the answer. A Pydantic model inherits `BaseModel`. Then the note becomes a field. Pydantic reads the answer, converts it when it can, and raises `ValidationError` when a field fails.

![Incoming values are checked, then stored or rejected](diagrams/check_fields.png)

Run the cells from top to bottom with **Shift+Enter**. Two cells define a class named `Student`. The second one replaces the first.

In this lesson you will:

- Watch a plain form reject `name=` and `age=`
- Store Issath, and see the text `"30"` become the number `30`
- Read the error when the name is a number and the age is `Khan`
- Add school rules for the name, the age, and the mark
- Attach an address, which is a model of its own


## 1. The form written as a normal class

`name: str` and `age: int` are notes on the class. Calling `StudentForm(name="Issath", age=30)` fails, because a normal class does not take those arguments.


In [1]:
class StudentForm:
    name: str
    age: int


try:
    StudentForm(name="Issath", age=30)
except TypeError as ex:
    print(ex)


StudentForm() takes no arguments


## 2. The same form as a model

`Student` inherits `BaseModel`, so `name` and `age` are fields. Both are required.

A form on a website often sends the age as text. `"30"` can be read as the whole number `30`, so that is what the model stores.


In [2]:
from pydantic import BaseModel, Field, ValidationError


class Student(BaseModel):
    name: str
    age: int


student = Student(name="Issath", age="30")
print(student.model_dump())
print(type(student.age))


{'name': 'Issath', 'age': 30}
<class 'int'>


## 3. An answer in the wrong field

`name=30` is a number, and the name must be text. `age="Khan"` is text that is not a whole number. `ValidationError` reports both fields.


In [3]:
try:
    Student(name=30, age="Khan")
except ValidationError as ex:
    print(ex)


2 validation errors for Student
name
  Input should be a valid string [type=string_type, input_value=30, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Khan', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing


## 4. School rules, and an address

The school now asks for three more things.

- The name must be 3 to 20 characters.
- The age must be from 13 to 60. `ge` means greater than or equal. `le` means less than or equal.
- The mark must be from 0 to 100. A missing mark is stored as `0`.

The address is its own small form: a city and a pincode. `Address` also inherits `BaseModel`, so those two answers are checked. `address: Address` places that form inside the student form. This class replaces the `Student` from section 2.

![An outer model can hold another model](diagrams/nested_model.png)


In [ ]:
class Address(BaseModel):
    city: str
    pincode: int


class Student(BaseModel):
    name: str = Field(min_length=3, max_length=20)
    age: int = Field(ge=13, le=60)
    mark: int = Field(ge=0, le=100, default=0)
    address: Address


## 5. A form the school sends back

`**answers` passes each dictionary key as an argument. `"ON"` is shorter than 3 characters. `"64"` becomes `64`, which is above 60. `"150"` becomes `150`, which is above 100.

The address is fine. `"600028"` becomes the integer `600028`, so that part of the form does not appear in the error.


In [4]:
answers = {
    "name": "ON",
    "age": "64",
    "mark": "150",
    "address": {"city": "chennai", "pincode": "600028"},
}
try:
    Student(**answers)
except ValidationError as ex:
    print(ex)


3 validation errors for Student
name
  String should have at least 3 characters [type=string_too_short, input_value='ON', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short
age
  Input should be less than or equal to 60 [type=less_than_equal, input_value='64', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
mark
  Input should be less than or equal to 100 [type=less_than_equal, input_value='150', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


## 6. A form the school accepts

Issath's form leaves the mark out, so the stored mark is `0`. Khan's form includes `50`. In both forms the pincode is stored as an integer.


In [5]:
issath = Student(
    name="Issath",
    age="30",
    address={"city": "chennai", "pincode": "600028"},
)
print(issath.model_dump())

khan = Student(
    name="Khan",
    age="35",
    mark="50",
    address={"city": "chennai", "pincode": "600028"},
)
print(khan.model_dump())
print(type(khan.address.pincode))


{'name': 'Issath', 'age': 30, 'mark': 0, 'address': {'city': 'chennai', 'pincode': 600028}}
{'name': 'Khan', 'age': 35, 'mark': 50, 'address': {'city': 'chennai', 'pincode': 600028}}
<class 'int'>


## Try it yourself

In the empty cell below:

1. Set Issath's age to `12` and read the error.
2. Change the pincode to `"chennai"` and read the error.
3. Add a `course` field with `min_length=3`, then store a student who studies Python.
